## Assignment 2 - Data Analysis - Web Mining

The web is an immense source of data, offering valuable insights across various domains. In this assignment, we will explore different ways to process, and analyze web data to uncover meaningful patterns and make informed decisions. Through five practical exercises, we will apply key data mining techniques to real-world scenarios.

Each exercise focuses on a specific application:

* *Exercise 1* - Clickbait classification
* *Exercise 2* - Sentiment analysis on comments
* *Exercise 3* - Movie recommendation
* *Exercise 4* - Association rules in online shopping
* *Exercise 5* - Clustering of mobile apps

These exercises will provide hands-on experience in working with real-world data collected from the web, helping you understand its potential for analysis.

For this assignment, complete all exercises that are marked in <span style='color:red;font-weight:bold'>red</span>. Please make sure all your cells run correctly (try to *Clear All Outputs* then *Run All* once before submitting). **Check the cells outputs are visibles even for the coding parts**

The assignment is due for <span style='color:red;font-weight:bold'>Thursday 26th of March 2026 at 23:59</span>.

No report is needed as all questions can be answered directly in this notebook file. You only need to give this notebook file completed on the [Moodle assignment page](https://moodle.msengineering.ch/mod/assign/view.php?id=229885). Only one file per group is required for submission.

If you have any questions or issues, please contact one of the assistants below:
- Cédric Campos Carvalho (*Teams* might be easier to discuss, mail: cedric.camposcarvalho@heig-vd.ch)
- Elena Najdenovska (mail: elena.najdenovska@heig-vd.ch)


Teacher : 
- Laura Elena Raileanu <Laura.Raileanu@heig-vd.ch>(mail: Laura.Raileanu@heig-vd.ch)

### Exercise 1 - Clickbait classification

The objective of this first part is to model a filter for "clickbait" in online news media. Clickbait headlines are designed to attract attention and drive clicks, often at the expense of accuracy or relevance.

To achieve this, we provide you with a dataset containing more than 10'000 press headlines collected in 2016. Each row in this dataset corresponds to a single headline, which is described by the following two attributes:

* `headline`: the text representing the title
* `clickbait`: the label identifying whether the title is a clickbait *(1)* or not *(0)*.

In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer

import warnings
warnings.simplefilter("ignore")

In [2]:
df_1 = pd.read_excel('data/part1_classification/news_clickbait.xlsx', engine="openpyxl")
df_1

,headline,clickbait
0,You Need To Tell Us If These Things Are Doughnuts,1
1,15 Great Pieces Of Relationship Advice From Books,1
2,Improved E-Mail Service From a Dedicated Device,0
3,"Two MBTA Green Line trains collide in Newton, ...",0
4,17 Struggles All Smartypants Will Understand,1
...,...,...
10536,Can You Match The Phone To The R&B Video,1
10537,19 Soul Food Recipes That Are Almost As Good A...,1
10538,16 Photos Of Desis That Will Give You Intense ...,1
10539,City Plans to Make Older Buildings Refit to Sa...,0


<p style='color:red;font-weight:bold'>Exercise 1.1 :</p>

The first step is to separate the dataset into two sets (training and test), complete the input parameters of `train_test_split` function.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df_1['headline'], df_1['clickbait'], test_size=0.2, random_state=42
)

<p style='color:red;font-weight:bold'>Exercise 1.2 :</p>

Create a pre-processing `Pipeline` using [scikit-learn Pipeline object](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).
In your pipeline you need:
- Vectorize your headlines with Term frequency-inverse document frequency.
- Transform your data in case of [`sparse matrix`](https://docs.scipy.org/doc/scipy/reference/sparse.html#module-scipy.sparse), so the data goes through the model without any issues.

**Do not forget to remove words giving no information for classification (i.e. [stop words](https://en.wikipedia.org/wiki/Stop_word)).**

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
])

<p style='color:red;font-weight:bold'>Exercise 1.3 :</p>

Create the pipeline using `make_pipeline`, combining the pre-processing pipeline and the `GaussianNB` model. Then, train the pipeline and find the score obtained with the test set.

In [5]:
pipeline_nb = make_pipeline(preprocessor, GaussianNB())

pipeline_nb.fit(X_train, y_train)

score = pipeline_nb.score(X_test, y_test)
print(f"Accuracy on test set: {score:.4f}")

Accuracy on test set: 0.8990


<p style='color:red;font-weight:bold'>Exercise 1.4 :</p>

Modify your split ratio for the training/testing set and see if there's a difference in the model's performances. Please explain your findings.

In [6]:
split_ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
results_split = []

for ratio in split_ratios:
    X_tr, X_te, y_tr, y_te = train_test_split(
        df_1['headline'], df_1['clickbait'], test_size=ratio, random_state=42
    )
    pipe = make_pipeline(preprocessor, GaussianNB())
    pipe.fit(X_tr, y_tr)
    acc = pipe.score(X_te, y_te)
    results_split.append({'test_size': ratio, 'train_samples': len(X_tr), 'test_samples': len(X_te), 'accuracy': round(acc, 4)})

df_split = pd.DataFrame(results_split)
print(df_split.to_string(index=False))

 test_size  train_samples  test_samples  accuracy
       0.1           9486          1055    0.8957
       0.2           8432          2109    0.8990
       0.3           7378          3163    0.8922
       0.4           6324          4217    0.8878
       0.5           5270          5271    0.8810


With a smaller test set (e.g. 10%), the model has more training data and may fit better, but the accuracy estimate is less reliable because it is evaluated on fewer samples. Conversely, a larger test set (e.g. 50%) gives a more robust evaluation but leaves less data for training, which can slightly decrease performance. Overall, GaussianNB is fairly stable across split ratios because TF-IDF features provide a good representation. The 20-30% range typically offers the best trade-off between having enough training data and a reliable evaluation.

<p style='color:red;font-weight:bold'>Exercise 1.5 :</p>

Keep your split validation but now incorporate [cross validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)). Use the [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) function and a K-Folding with 5 splits over the training set without using the test set. Then, calculate the averaged accuracy (with standard deviation) for 5 folds.

**Explain the results obtained and how to read them compared to the split validation.**

In [7]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipeline_nb, X_train, y_train, cv=kf, scoring='accuracy')

print(f"Cross-validation scores (5 folds): {cv_scores}")
print(f"Mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

Cross-validation scores (5 folds): [0.88322466 0.89211618 0.89620403 0.8831554  0.90154211]
Mean accuracy: 0.8912 (+/- 0.0072)


Cross-validation provides a more robust estimate of model performance than a single train/test split. Instead of relying on one particular partition, it trains and evaluates the model 5 times on different subsets of the training data. The mean accuracy gives a more stable performance estimate, while the standard deviation indicates how much the performance varies across folds. A low standard deviation means the model generalizes consistently. Compared to the single split validation, cross-validation reduces the risk of getting an overly optimistic or pessimistic score due to a lucky or unlucky data split.

<p style='color:red;font-weight:bold'>Exercise 1.6 :</p>

Try to use atleast 3 different classifiers and report their results in a table. Compare the results between them.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

classifiers = {
    'GaussianNB': GaussianNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'LinearSVC': LinearSVC(max_iter=2000),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results_clf = []
for name, clf in classifiers.items():
    pipe = make_pipeline(preprocessor, clf)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='accuracy')
    pipe.fit(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    results_clf.append({
        'Classifier': name,
        'CV Mean Accuracy': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
        'Test Accuracy': round(test_acc, 4),
    })

df_classifiers = pd.DataFrame(results_clf)
df_classifiers

,Classifier,CV Mean Accuracy,CV Std,Test Accuracy
0,GaussianNB,0.8912,0.0072,0.8990
1,LogisticRegression,0.9336,0.0027,0.9279
2,LinearSVC,0.9387,0.0029,0.9388
3,RandomForest,0.8918,0.0052,0.8933


Linear models like **LogisticRegression** and **LinearSVC** tend to perform very well on text classification tasks because TF-IDF features create a high-dimensional, linearly separable space. They typically outperform GaussianNB, which assumes feature independence and a Gaussian distribution — assumptions that do not hold well for sparse TF-IDF vectors.

**RandomForest** can capture non-linear patterns but tends to be slower and may not improve over linear models on this type of high-dimensional sparse data. Overall, LinearSVC and LogisticRegression are expected to achieve the highest accuracy for clickbait detection, while GaussianNB provides a reasonable but weaker baseline.

<p style='color:red;font-weight:bold'>Exercise 1.7 :</p>

In text processing we often use a stemming step for the pre-processing, explain what it consists and how it can be useful then give an example.

**Stemming** is a text pre-processing technique that reduces words to their root form (called the *stem*) by removing suffixes and prefixes. Unlike lemmatization, stemming uses heuristic rules and does not necessarily produce a valid dictionary word.

**How it is useful:** Stemming reduces the vocabulary size by grouping together words that share the same root. This helps the model treat different forms of the same word as equivalent, improving generalization and reducing feature sparsity. For example, without stemming, "running", "runs", and "runner" would be three separate features, but with stemming they all become "run".

**Example using the Porter Stemmer:**

| Original word | Stemmed form |
|---|---|
| running | run |
| studies | studi |
| classification | classif |
| easily | easili |
| clickbaiting | clickbait |

As we can see, stemming is aggressive and can sometimes produce non-dictionary words (e.g., "studi", "easili"), but it effectively groups related words together for the purpose of text classification.

### Exercise 2 - Sentiment analysis on comments

The goal of the second part is to analyze tweets from the COVID-19 period and perform sentiment analysis to determine whether they express a positive or negative sentiment.

In the first step, we will work with the dataset `CoronaTwitterComments_2labels.xlsx`, which contains approximately 3'000 comments related to COVID-19 from Twitter in March 2020. These comments are already labeled with *1* (positive comment) or *-1* (negative comment). In the second step, we will consider the dataset `CoronaTwitterComments_3labels.xlsx`, which includes additional neutral comments (labeled with *0*). You will see how we can process the text data with the help of [WordNet](https://wordnet.princeton.edu/) to retrieve a sentiment and evaluate the results.

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

import nltk

In [10]:
# NLTK packages needed for this exercise, feel free to add some if you need it.

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('sentiwordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\proky\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\proky\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\proky\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package sentiwordnet to
[nltk_data]     C:\Users\proky\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\sentiwordnet.zip.


True

In [11]:
corona_tweets2 = pd.read_excel('data/part2_sentimentanalysis/CoronaTwitterComments_2labels.xlsx', engine="openpyxl")
corona_tweets3 = pd.read_excel('data/part2_sentimentanalysis/CoronaTwitterComments_3labels.xlsx', engine="openpyxl")

print(corona_tweets2.SentimentLabel.unique(), corona_tweets3.SentimentLabel.unique())
corona_tweets2

[-1  1] [-1  1  0]


,OriginalTweet,SentimentLabel
0,TRENDING: New Yorkers encounter empty supermar...,-1
1,When I couldn't find hand sanitizer at Fred Me...,1
2,Find out how you can protect yourself and love...,1
3,#Panic buying hits #NewYork City as anxious sh...,-1
4,Voting in the age of #coronavirus = hand sanit...,1
...,...,...
3174,"@RicePolitics @MDCounties Craig, will you call...",-1
3175,Meanwhile In A Supermarket in Israel -- People...,1
3176,Did you panic buy a lot of non-perishable item...,-1
3177,Gov need to do somethings instead of biar je r...,-1


<p style='color:red;font-weight:bold'>Exercise 2.1 :</p>

The first step is to pre-process the text (like in previous exercise) to later use a WordNet sentiment analysis over it.

The class inherits from `BaseEstimator` and `TransformerMixin`, ensuring seamless integration with other scikit-learn modules. This allows it to utilize the familiar `fit`, `transform`, and `fit_transform` functions of the library.

Your task is to update the `NLTKPreprocessor` class with the needed functions to be used later in the `transform` function.
- Add a tokenizer specialized in tweets.
- Remove stop words and other unuseful characters.
- Transform in lowercase the text
- Lemmatize the text using for Wordnet.

*Advice : If you encounter any issues, refer to the `apply_pipeline` function to understand how it works. This function should **not** be modified.*

In [12]:
import string
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords  

class NLTKPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
                  
        self.pipeline = [
            self.tweet_tokenizer,
            self.stop_words,
            self.remove_punctuation,
            self.tweet_lemmatizer
        ]
        
    def apply_pipeline(self, x):
        for transform in self.pipeline:
            x = transform(x)
        return x
  
    def fit(self, X, y=None):
        self.fitted_ = True
        return self

    def transform(self, X):
        return [self.apply_pipeline(x) for x in X]
    
    def tweet_tokenizer(self, x):
        tokenizedtweet = TweetTokenizer(preserve_case=False)
        return tokenizedtweet.tokenize(x)

    def remove_punctuation(self, x):
        translator = str.maketrans('', '', string.punctuation)
        string_withoutpunctuation = []
        
        for word in x:
            if word.startswith("http://") or word.startswith("https://") or word.startswith("www."):
                continue
                
            cleaned_word = word.translate(translator)
            
            if cleaned_word:
                string_withoutpunctuation.append(cleaned_word)
            
        return string_withoutpunctuation
    
    def stop_words(self, x):
        stop_words = set(stopwords.words('english'))  
        
        # Filter out stopwords from the tokenized words  
        # enlever le lower? 
        filtered_words = [word for word in x if word not in stop_words]  
        return filtered_words
    
    def tweet_lemmatizer(self, x):
        lemmatizer = WordNetLemmatizer()
        return [lemmatizer.lemmatize(word) for word in x]

<p style='color:red;font-weight:bold'>Exercise 2.2 :</p>

The next step is to create the `WordNetSentimentAnalyzer`. You will need to create three functions to add to your pipeline : 
1. The WordNet sentiment analysis requires to have the tag of each word of the sentence. Tagging in part of speech (POS) is the process of assigning grammatical categories, such as nouns, verbs, or adjectives, to words in a text based on their role in a sentence. `nltk` has a pre-trained model that can tag these words, find it and apply it to the pipeline.
2. For WordNet the tagging is different from the pre-trained `nltk` model. Create a function replacing the tags with the WordNet tags using `nltk.corpus.wordnet` module.
3. Create the sentiment function which sums the positive sentiment ($\text{pos}$) and negative sentiment ($\text{neg}$) of each word (using `nltk.corupus.sentiwordnet` module). The value needs to be taken in account only if it is superior to a certain threshold (i.e. $\text{treshold} = 0.05$). Then, return a single value representing the sentiment of the sentence such as it is $\text{sentiment} < 0$ if it's negative and $\text{sentiment} > 0$ if positive.


In [13]:
from nltk.corpus import wordnet as wn
from nltk.corpus import sentiwordnet as swn

class WordNetSentimentAnalyzer(BaseEstimator, TransformerMixin):
    def __init__(self):

        self.pipeline = [
            self.wordnet_tag_convert,
            self.sentiment_analyzer
        ]

    def apply_pipeline(self, x):
        for transform in self.pipeline:
            x = transform(x)
        return x
    
    def fit(self, X, y=None):
        self.fitted_ = True
        return self
    
    def transform(self, X):
        return [self.apply_pipeline(x) for x in X]
    
    def wordnet_tag_convert(self, x):
         tagged = nltk.pos_tag(x)
         converted_tags = []
         
         for (word, tag) in tagged:
             if tag.startswith('J'):
                 tag = 'a'
             elif tag.startswith('V'):
                 tag = 'v'
             elif tag.startswith('N'):
                 tag = 'n'
             elif tag.startswith('R'):
                 tag = 'r'
             else :
                 tag = 'n' 
            
             converted_tags.append((word, tag)) 
         
         return converted_tags
    
    def sentiment_analyzer(self, x):
        sentimentPositive = 0.0
        sentimentNegatve = 0.0

        for (word, tag) in x :
            if tag is None :
                continue
            
            wordSynst = wn.synsets(word, pos=tag)

            if not wordSynst:
                continue  
                
            chosenSynst = wordSynst[0]
            sentiWordNet = swn.senti_synset(chosenSynst.name())
                        
            if sentiWordNet.pos_score() > 0.05 :
                sentimentPositive += sentiWordNet.pos_score() 
            
            if sentiWordNet.neg_score() > 0.05:
                sentimentNegatve += sentiWordNet.neg_score()
            
        
        sentenceScore = sentimentPositive - sentimentNegatve
          
        return sentenceScore

<p style='color:red;font-weight:bold'>Exercise 2.3 :</p>

Create two functions that return the sentiment class to compare it to the labels.
* `get_sentiment_class_2` : For positive and negative only.
* `get_sentiment_class_3` : For positive, negative and neutral.

**Do not forget that it is going to be used in a `scikit-learn` `Pipeline`!**

In [14]:
from sklearn.pipeline import Pipeline

def get_sentiment_class_2(sent_val):
    return 1 if sent_val > 0 else -1

def get_sentiment_class_3(sent_val):
    if sent_val > 0:
        return 1
    elif sent_val < 0:
        return -1
    else:
        return 0

class SentimentClass2Transformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [get_sentiment_class_2(x) for x in X]
    
class SentimentClass3Transformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [get_sentiment_class_3(x) for x in X]

<p style='color:red;font-weight:bold'>Exercise 2.4 :</p>

Create a `scikit-learn` `Pipeline` using the 3 previous *Transformers* that you created (using the 2 class sentiment).

Then, complete the function `get_sentiment_tweet` and test it with your own Tweet (**Do not change its signature !**).

In [15]:
pipeline_2 = Pipeline(steps=[
    ("preprocess", NLTKPreprocessor()),
    ("sentiment_score", WordNetSentimentAnalyzer()),
    ("sentiment_class_2", SentimentClass2Transformer())
])

pipeline_3 = Pipeline(steps=[
    ("preprocess", NLTKPreprocessor()),
    ("sentiment_score", WordNetSentimentAnalyzer()),
    ("sentiment_class_3", SentimentClass3Transformer())
])

def get_sentiment_tweet(tweet):
    return pipeline_2.fit_transform([tweet])[0]

In [16]:
my_tweet = "It is a very bad day today, it is raining outside"
print(get_sentiment_tweet(my_tweet))

-1


<p style='color:red;font-weight:bold'>Exercise 2.5 :</p>

Compute the accuracy obtained with all tweets of the dataset. Then, do the same for the 3 labels class.

In [17]:
from sklearn.metrics import accuracy_score

# Accuracy 
y_pred_2 = pipeline_2.fit_transform(corona_tweets2["OriginalTweet"])
y_true_2 = corona_tweets2["SentimentLabel"]

acc_2 = accuracy_score(y_true_2, y_pred_2)
print("Accuracy 2 classes :", acc_2)

y_pred_3 = pipeline_3.fit_transform(corona_tweets2["OriginalTweet"])
y_true_3 = corona_tweets2["SentimentLabel"]

acc_3 = accuracy_score(y_true_3, y_pred_3)
print("Accuracy 3 classes :", acc_3)

Accuracy 2 classes : 0.6517772884554891
Accuracy 3 classes : 0.5929537590437245


<p style='color:red;font-weight:bold'>Exercise 2.6 :</p>

Analyze the results of each pre-processing (`NLTKPreprocessor`) step to try to understand what you could do to improve it. For example, what could you add into this pipeline ? *(No implementation needed)*

In [18]:
#Print results in each step of the NLTKPreprocessor to analyze what happens
 
tweets = corona_tweets2["OriginalTweet"].head(10)

proc = NLTKPreprocessor()

for i, tweet in enumerate(tweets, start=1):
    print(f"\n--- Tweet {i} ---")
    print("Original:", tweet)

    step1 = proc.tweet_tokenizer(tweet)
    print("Tokenizer:", step1)

    step2 = proc.stop_words(step1)
    print("Stop words:", step2)

    step3 = proc.remove_punctuation(step2)
    print("No punctuation:", step3)



--- Tweet 1 ---
Original: TRENDING: New Yorkers encounter empty supermarket shelves (pictured, Wegmans in Brooklyn), sold-out online grocers (FoodKick, MaxDelivery) as #coronavirus-fearing shoppers stock up https://t.co/Gr76pcrLWh https://t.co/ivMKMsqdT1
Tokenizer: ['trending', ':', 'new', 'yorkers', 'encounter', 'empty', 'supermarket', 'shelves', '(', 'pictured', ',', 'wegmans', 'in', 'brooklyn', ')', ',', 'sold-out', 'online', 'grocers', '(', 'foodkick', ',', 'maxdelivery', ')', 'as', '#coronavirus-fearing', 'shoppers', 'stock', 'up', 'https://t.co/Gr76pcrLWh', 'https://t.co/ivMKMsqdT1']
Stop words: ['trending', ':', 'new', 'yorkers', 'encounter', 'empty', 'supermarket', 'shelves', '(', 'pictured', ',', 'wegmans', 'brooklyn', ')', ',', 'sold-out', 'online', 'grocers', '(', 'foodkick', ',', 'maxdelivery', ')', '#coronavirus-fearing', 'shoppers', 'stock', 'https://t.co/Gr76pcrLWh', 'https://t.co/ivMKMsqdT1']
No punctuation: ['trending', 'new', 'yorkers', 'encounter', 'empty', 'superma

# My analysis of possible improvements

### First observation

Numbers appear in several tweets (e.g., prices, quantities, phone numbers), but they generally do not contribute significantly to sentiment analysis. They may introduce noise and reduce the effectiveness of lexical-based methods such as WordNet or SentiWordNet. Therefore, removing or filtering numerical values could help improve the overall quality of the sentiment evaluation. However, it is worth noting that some numbers may carry contextual meaning (e.g., “1st confirmed case”), so a more selective approach might be preferable.

### Second observation

Tweets often contain informal language, abbreviations, and contractions (e.g., “btw”, “whats”, “lets”), which are not always properly handled by the current preprocessing pipeline. These forms may not be recognized correctly by lexical resources such as WordNet, leading to information loss or inaccurate sentiment scores. Introducing a normalization step to expand abbreviations and handle informal expressions could improve the quality and consistency of the sentiment analysis.

### Exercise 3 - Movie recommendation

The objective of this exercise is to create a recommendation system of movies based on the user ratings. We will focus on the collaborative approach for movie recommendation using the provided dataset, which contains approximately 9'800 movies rated by 610 users from MovieLens.

More specifically, the `movies.csv` dataset, which describes the movies, includes three attributes:

* `movieId`: unique identifier of the movie
* `title`: the title of the movie (with the release year in parentheses)
* `genres`: the genres of the movie

The `ratings.csv` dataset, which contains user ratings for the movies, includes four attributes:

* `userId`: unique identifier of the user
* `movieId`: unique identifier of the movie
* `rating`: the user's rating for the corresponding movie
* `timestamp`: the timestamp of the rating

In [19]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd


In [20]:
df_movies = pd.read_csv('data/part3_recommandationdesfilms/movies.csv')
df_ratings = pd.read_csv('data/part3_recommandationdesfilms/ratings.csv')

df_3 = pd.merge(df_movies, df_ratings, on='movieId')
df_3

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483
...,...,...,...,...,...,...
100831,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,184,4.0,1537109082
100832,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,184,3.5,1537109545
100833,193585,Flint (2017),Drama,184,3.5,1537109805
100834,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,184,3.5,1537110021


<p style='color:red;font-weight:bold'>Exercise 3.1 :</p>

Use `df_3` to create a second `DataFrame` with the rating of every movies for each `userId`. If the user never watched a movie the value is $0$. 
For this step, use the [`pivot` function](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html).

In [21]:
user_item_matrix = df_3.pivot(index="userId", columns="movieId", values="rating")
# I sub 2.5 to all vals so a 5 have a positive action and a 1 has a negative action. 
# 0 should be neutral at the center instead of being negative
user_item_matrix = user_item_matrix.sub(2.5).fillna(0) # TODO 3.1
X = user_item_matrix.to_numpy()
X

array([[ 1.5,  0. ,  1.5, ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       ...,
       [ 0. , -0.5, -0.5, ...,  0. ,  0. ,  0. ],
       [ 0.5,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 2.5,  0. ,  0. , ...,  0. ,  0. ,  0. ]], shape=(610, 9724))

<p style='color:red;font-weight:bold'>Exercise 3.2 :</p>

First separate the data using `train_test_split` function with $20\%$ of the data in the test set.
Create an `knn` model using the [`NearestNeighbors`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html#sklearn.neighbors.NearestNeighbors) model from `scikit-learn`. For now use $N=80$, and the `cosine` metric, then train it.

**Explain in few sentences how does the `cosine` metric works by taking our case in an example (include the formula).**

---

$$
\text{cosine\_similarity}(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \, \|\mathbf{B}\|}
$$

The cosine metric works by comparing the angle between two vectors. This goes from 1 (same angle) to -1 (opposed direction) passing by 0 (ortogonal). Magnitude has no influence on the result.


We have 4 users who rated 2 movies on a scale from 1 to 5.

- Movie A: `[5, 4, 0, 1]`
- Movie B: `[4, 0, 0, 1]`

`0` means that the user did not rate the movie.

### Step 1: Compute the dot product

$$
A \cdot B = (5 \times 4) + (4 \times 0) + (0 \times 0) + (1 \times 1)
$$

$$
A \cdot B = 20 + 0 + 0 + 1 = 21
$$


### Step 2: Compute the norm of Movie A

$$
\|A\| = \sqrt{5^2 + 4^2 + 0^2 + 1^2}
$$

$$
\|A\| = \sqrt{25 + 16 + 0 + 1}
$$

$$
\|A\| = \sqrt{42}
$$

### Step 3: Compute the norm of Movie B

$$
\|B\| = \sqrt{4^2 + 0^2 + 0^2 + 1^2}
$$

$$
\|B\| = \sqrt{16 + 0 + 0 + 1}
$$

$$
\|B\| = \sqrt{17}
$$

### Step 4: Apply the cosine similarity formula

$$
\text{cosine\_similarity}(A, B) =
\frac{21}{\sqrt{42} \cdot \sqrt{17}}  \approx 0.79
$$


So, Movie A and Movie B have a cosine similarity of about **0.79**.

---

In [22]:
movie_train, movie_test = train_test_split(user_item_matrix, test_size=0.2)

knn = NearestNeighbors(n_neighbors=80, metric="cosine")

knn.fit(movie_train.values)

print(movie_test.shape)
print(movie_train.shape)

(122, 9724)
(488, 9724)


<p style='color:red;font-weight:bold'>Exercise 3.3 :</p>

Create the `predict` function by using the previous trained model.

*Hints : Think about the model returns and what they represent. Then, use the closest indexes to obtain information on the best matching movies.* 

In [23]:
def predict(user_ratings, user_item_matrix, knn):
    user_vector = user_ratings.values.reshape(1, -1)
    
    distances, indices = knn.kneighbors(user_vector)
    
    neighbor_ratings = user_item_matrix.iloc[indices.flatten()]
    
    result = neighbor_ratings.sum(axis=0)
    
    return result

<p style='color:red;font-weight:bold'>Exercise 3.4 :</p>

Finally, complete the `prec_at_k` function, which measures the precision of our model. This metric calculates how many of the top $k$ movies recommended to a user are actually relevant. A movie is considered relevant if the user has given it a score of at least 4..

In [24]:
def prec_at_k(pred_Y, user_rating, k):
    values = pred_Y.to_numpy()   # only the values
    
    # get the index of the top k max values
    top_k_pos = np.argpartition(values, -k)[-k:]
    #print("positions:", top_k_pos)
    #print("top-k values:", pred_Y.iloc[top_k_pos])
    #print("user  values:", user_rating.iloc[top_k_pos])
    
    relevance = 0
    for val in user_rating.iloc[top_k_pos]:
        # 4 - shift of 2.5 = 1.5
        if val >= 1.5:
            relevance += 1
    
    return relevance / k

In [25]:
user_id = 10
pred = predict(movie_test.iloc[user_id], user_item_matrix, knn)

prec_at_k(pred, movie_test.iloc[user_id], 5)

0.4

<p style='color:red;font-weight:bold'>Exercise 3.5 :</p>

Now, use the function with three different values for k ($5$, $15$, $25$) each time. Then, use different $N$ neighbours and a different split ratio.

Report all your results bellow and explain the impact of these hyperparameters in the model.

In [26]:
def get_results(test_set, k, user_item_matrix, knn):
    user_ids_test = list(test_set.index)
    total_score = 0

    #print("user_id in test set", user_ids_test)

    for user_id in user_ids_test:
        pred = predict(test_set.loc[user_id], user_item_matrix, knn)
        total_score += prec_at_k(pred, test_set.loc[user_id], k)

    return total_score / len(user_ids_test)

k_to_test = [2,5,15,25]
N_to_test = [40,80,120]
Split_to_test = [0.1,0.2,0.5]

import csv

with open("result_e3.csv", "w", newline="\n") as result_file:
    
    writer = csv.writer(result_file, delimiter=',',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)
    
    writer.writerow(['score', 'N', 'Split', 'k'])

    for N in N_to_test:
        knn = NearestNeighbors(n_neighbors=N, metric="cosine")
        for split in Split_to_test:
            movie_train, movie_test = train_test_split(user_item_matrix, test_size=split)
            knn.fit(movie_train.values)
            for k in k_to_test:
                res = get_results(movie_test, k, user_item_matrix, knn)
                writer.writerow([res, N, split, k])
                print(res, N, split, k)
        

0.3524590163934426 40 0.1 2
0.37377049180327865 40 0.1 5
0.33551912568305997 40 0.1 15
0.30098360655737705 40 0.1 25
0.3770491803278688 40 0.2 2
0.32786885245901654 40 0.2 5
0.2606557377049179 40 0.2 15
0.23737704918032762 40 0.2 25
0.4065573770491803 40 0.5 2
0.3593442622950822 40 0.5 5
0.2800000000000002 40 0.5 15
0.2436721311475407 40 0.5 25
0.4426229508196721 80 0.1 2
0.39344262295081966 80 0.1 5
0.33005464480874314 80 0.1 15
0.2944262295081966 80 0.1 25
0.4344262295081967 80 0.2 2
0.3967213114754101 80 0.2 5
0.32021857923497254 80 0.2 15
0.27934426229508186 80 0.2 25
0.5098360655737705 80 0.5 2
0.4059016393442625 80 0.5 5
0.29967213114754115 80 0.5 15
0.2670163934426227 80 0.5 25
0.48360655737704916 120 0.1 2
0.41967213114754087 120 0.1 5
0.3289617486338798 120 0.1 15
0.27934426229508197 120 0.1 25
0.38524590163934425 120 0.2 2
0.38688524590163936 120 0.2 5
0.3213114754098361 120 0.2 15
0.28688524590163916 120 0.2 25
0.46229508196721314 120 0.5 2
0.4288524590163937 120 0.5 5
0.330

In [27]:
df_result = pd.read_csv('result_e3.csv')
df_result

,score,N,Split,k
0,0.352459,40,0.1,2
1,0.373770,40,0.1,5
2,0.335519,40,0.1,15
3,0.300984,40,0.1,25
4,0.377049,40,0.2,2
5,0.327869,40,0.2,5
6,0.260656,40,0.2,15
7,0.237377,40,0.2,25
8,0.406557,40,0.5,2
9,0.359344,40,0.5,5


We can see that the average precision of our recommandation model is better when recommanding less movies (k small) which makes sense since the k + 1 movie recommanded will have a worse score than the k movie.

Something interesting is the model is better when the knn has a neighbourg of 80 : the 4 highest score are made using 80 as N.

The split is seems to yield the highest result when using a 20% split but difference are small. We would guess that the optimal split is data depend.

### Exercise 4 - Association rules in online shopping

In this fourth part, we will focus on a **Market Basket Analysis** problem. The dataset provided contains **online sales transactions** from an e-commerce site over one year. Our goal is to generate **association rules** based on these sales to identify which items are frequently bought together. You can refer to the source mentioned in the **README file** included with the data for more details.  

The dataset is structured as follows, with each row representing the details of a product sale:  
- `InvoiceNo`: invoice/sale identifier  
- `StockCode`: product identifier  
- `Description`: purchased product  
- `Quantity`: quantity sold  
- `InvoiceDate`: order/payment date  
- `UnitPrice`: price of the product  
- `CustomerID`: customer identifier  
- `Country`: customer’s country of residence

In [28]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

In [29]:
df_4 = pd.read_excel('data/part4_marketbasketanalysis/OnlineRetailDataset.xlsx', engine='openpyxl')
df_4

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
397919,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France
397920,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France
397921,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France
397922,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France


<p style='color:red;font-weight:bold'>Exercise 4.1 :</p>

Transform the given dataset into a one-hot encoded format for association rule mining. Group items by customer so that each row represents a unique customer with a list of purchased items. Then, convert this into a binary matrix where each column is an item, and values indicate whether a customer bought that item (1) or not (0). Use [`MultiLabelBinarizer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MultiLabelBinarizer.html) to achieve this.

In [30]:
# TODO 4.1
grouped = df_4.groupby("CustomerID")["StockCode"].apply(list)

print(grouped)

mlb = MultiLabelBinarizer()
onehot = mlb.fit_transform(grouped)
print(list(mlb.classes_))

one_hot_df = pd.DataFrame(onehot, index=grouped.index, columns=mlb.classes_, dtype=bool)

print(one_hot_df)

CustomerID
12346                                              [23166]
12347    [85116, 22375, 71477, 22492, 22771, 22772, 227...
12348    [84992, 22951, 84991, 84991, 21213, 21213, 226...
12349    [23112, 23460, 21564, 21411, 21563, 22131, 221...
12350    [21908, 22412, 79066K, 79191C, 22348, 84086C, ...
                               ...                        
18280    [82484, 22180, 22467, 22725, 22727, 22495, 223...
18281    [22037, 22716, 22028, 23007, 23008, 23209, 22467]
18282    [21270, 23187, 23295, 22089, 21108, 21109, 224...
18283    [22356, 20726, 22384, 22386, 20717, 20718, 850...
18287    [22755, 22754, 22753, 22756, 22758, 22757, 227...
Name: StockCode, Length: 4339, dtype: object
['10002', '10080', '10120', '10123C', '10124A', '10124G', '10125', '10133', '10135', '11001', '15030', '15034', '15036', '15039', '15044A', '15044B', '15044C', '15044D', '15056BL', '15056N', '15056P', '15058A', '15058B', '15058C', '15060B', '16008', '16010', '16011', '16012', '16014', '16015', 

<p style='color:red;font-weight:bold'>Exercise 4.2 :</p>

Finally, create the association rules based on the frequent patterns obtained through `FP-growth` algorithm. Check [`mlxtend`](https://rasbt.github.io/mlxtend/) documentation to complete this task. Then, show what are the association rules with a minimum of $0.9$ confidence.

In [31]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

frequent_itemsets = fpgrowth(one_hot_df, min_support=0.03, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.9)

print(rules)

                     antecedents consequents  antecedent support  \
0                 (22697, 23245)     (22699)            0.036875   
1                        (22698)     (22697)            0.073519   
2                 (22698, 22699)     (22697)            0.062687   
3                 (22423, 22698)     (22697)            0.059691   
4          (22423, 22697, 22698)     (22699)            0.056465   
..                           ...         ...                 ...   
89  (20725, 20728, 23207, 20727)     (22382)            0.032496   
90  (20725, 20727, 23207, 22382)     (20728)            0.033418   
91                (22386, 23202)    (85099B)            0.035262   
92                (22386, 23209)     (23203)            0.032496   
93                (22577, 22579)     (22578)            0.039640   

    consequent support   support  confidence       lift  representativity  \
0             0.097488  0.033418    0.906250   9.296025               1.0   
1             0.089191  0.068

<p style='color:red;font-weight:bold'>Exercise 4.3 :</p>

Do you observe any changes in the different parameters of the *FP-Growth* and *Association Rules* functions ? Please comment on the chosen parameters and the results obtained.

In [32]:
# TODO 4.3

niche = fpgrowth(one_hot_df, min_support=0.02, use_colnames=True)

rules_niche = association_rules(niche, metric="confidence", min_threshold=0.9)

print(rules_niche)


                 antecedents consequents  antecedent support  \
0             (22697, 23245)     (22699)            0.036875   
1      (22423, 22697, 23245)     (22699)            0.031805   
2             (22697, 22960)     (22699)            0.025582   
3      (22423, 22697, 22960)     (22699)            0.022586   
4      (22697, 22960, 22699)     (22423)            0.023738   
...                      ...         ...                 ...   
5739          (82581, 82578)     (82580)            0.024430   
5740   (23583, 22383, 20725)     (20728)            0.023047   
5741  (23583, 85099B, 23209)     (23203)            0.021894   
5742   (23583, 23203, 20725)     (23209)            0.021434   
5743                 (23681)     (23209)            0.024199   

      consequent support   support  confidence       lift  representativity  \
0               0.097488  0.033418    0.906250   9.296025               1.0   
1               0.097488  0.029500    0.927536   9.514373               1

In [33]:

general = fpgrowth(one_hot_df, min_support=0.04, use_colnames=True)

rules_general = association_rules(general, metric="confidence", min_threshold=0.9)

print(rules_general)

             antecedents consequents  antecedent support  consequent support  \
0                (22698)     (22697)            0.073519            0.089191   
1         (22698, 22699)     (22697)            0.062687            0.089191   
2         (22423, 22698)     (22697)            0.059691            0.089191   
3  (22423, 22697, 22698)     (22699)            0.056465            0.097488   
4  (22423, 22698, 22699)     (22697)            0.053699            0.089191   
5         (22728, 22726)     (22727)            0.044941            0.089422   
6                (22804)    (85123A)            0.058078            0.197280   

    support  confidence       lift  representativity  leverage  conviction  \
0  0.068218    0.927900  10.403506               1.0  0.061661   12.632524   
1  0.060152    0.959559  10.758464               1.0  0.054561   22.521821   
2  0.056465    0.945946  10.605838               1.0  0.051141   16.849965   
3  0.051625    0.914286   9.378453             

In [34]:
rules_general_low_conf = association_rules(general, metric="confidence", min_threshold=0.75)

print(rules_general_low_conf)

        antecedents consequents  antecedent support  consequent support  \
0           (22699)     (22423)            0.097488            0.203042   
1           (22697)     (22699)            0.089191            0.097488   
2           (22699)     (22697)            0.097488            0.089191   
3           (22697)     (22423)            0.089191            0.203042   
4    (22423, 22697)     (22699)            0.068449            0.097488   
..              ...         ...                 ...                 ...   
126         (22386)    (85099B)            0.085734            0.146347   
127  (22386, 23203)    (85099B)            0.045863            0.146347   
128         (23300)     (23301)            0.080894            0.093339   
129         (22578)     (22577)            0.076515            0.079050   
130         (22579)     (22578)            0.052086            0.076515   

      support  confidence       lift  representativity  leverage  conviction  \
0    0.073980    0.

*TODO 4.3*

---

### FP-Growth

For FP-Growth, min_support represents the minimum proportion of transactions in which an itemset must appear to be kept.

A high support means keeping only very common itemsets, such as (bread, butter), which represent general purchasing patterns.

A low support captures rarer itemsets, such as (champagne, sabre), which correspond to more specific or niche patterns.

Therefore, min_support is a threshold that determines whether we keep only general itemsets (high min_support) or also include more specific ones (low min_support). Also, a high min_support reduces the computationnal cost.

In the code, 'general' is way faster than 'niche' because fewer item combinations need to be explored.

### Association rule

In the code, 'rules_general' has way less rules than 'rules_niche' because it focuses on the most frequent pattern only. 

Using a high confidence threshold like 0.9 ensures that only strong rules are chosen. Using a lower threshold will increase the number of rules but make them less reliable. In the code, 'rules_general_low_conf' has more rules than its equivalent with higher confidence because of the lower minimal confidence.


Also, other metrics are possible to select (since confidence could be misleading), which are more specialised criteria mostly based mathematically on the base ones (see [doc](https://rasbt.github.io/mlxtend/api_subpackages/mlxtend.frequent_patterns/#association_rules) for context)

The following document seems a relevant ressource for choosing a good metric :

>Michael Hahsler, A Probabilistic Comparison of Commonly Used Interest Measures for Association Rules, 2015, URL: https://mhahsler.github.io/arules/docs/measures

---

<p style='color:red;font-weight:bold'>Exercise 4.4 :</p>

Is it possible to use other column(s) from the initial data to generate interesting rules?

*TODO 4.4*

---

We could use price or quantity if we discretise the values first.

The price could be transformed into discrete categories to analyse whether cheaper products are bought together or if expensive items are associated with specific combinations.

The quantity could be used in the same way between single purchases and bulk purchases. This would allow us to identify patterns such as customers who tend to buy large quantities of certain items together.

---

### Exercise 5 - Clustering of mobile apps

In this final part, we will perform clustering of applications from the Google Play Store. You can refer to the source mentioned in the **README file** included with the data for more details.

Among the attributes available in the `googleplaystore.xlsx` file, we will use the following:

* `Rating`: overall user rating of the application
* `Reviews`: number of user reviews
* `Size`: size of the application
* `Installs`: number of users who installed the application
* `Price`: price of the application



In [35]:
from sklearn.cluster import KMeans
import seaborn as sns
import pandas as pd
import numpy as np

In [36]:
df_5 = pd.read_excel('data/part5_clustering/googleplaystore.xlsx', engine='openpyxl')
df_5

,Rating,Reviews,Size,Installs,Price,App,Category
0,4.1,159,19.0,10000,0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN
1,3.9,967,14.0,500000,0,Coloring book moana,ART_AND_DESIGN
2,4.7,87510,8.7,5000000,0,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN
3,4.5,215644,25.0,50000000,0,Sketch - Draw & Paint,ART_AND_DESIGN
4,4.3,967,2.8,100000,0,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN
...,...,...,...,...,...,...,...
9361,4.0,7,2.6,500,0,FR Calculator,FAMILY
9362,4.5,38,53.0,5000,0,Sya9a Maroc - FR,FAMILY
9363,5.0,4,3.6,100,0,Fr. Mike Schmitz Audio Teachings,FAMILY
9364,4.5,114,0.0,1000,0,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE


<p style='color:red;font-weight:bold'>Exercise 5.1 :</p>

Select the numerical features of the dataset then standardize them by removing the mean and scaling to unit variance. 
Then, do a first training using [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) algorithm with $5$ clusters.

In [37]:
from sklearn.preprocessing import StandardScaler

numerical_features = ['Rating', 'Reviews', 'Size', 'Installs', 'Price']
X_5 = df_5[numerical_features].values

scaler = StandardScaler()
X_5_scaled = scaler.fit_transform(X_5)

kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_5.fit(X_5_scaled)

df_5['Cluster'] = kmeans_5.labels_
print(f"Cluster distribution:\n{df_5['Cluster'].value_counts().sort_index()}")

Cluster distribution:
Cluster
0    7513
1      15
2     136
3     138
4    1564
Name: count, dtype: int64


<p style='color:red;font-weight:bold'>Exercise 5.2 :</p>

Complete the function `average_cluster_distance` to compute a clustering performance metric. It should return a list where the first value is the global average distance from all samples to their assigned centroids, followed by the average distances per centroid. 

In [38]:
def average_cluster_distance(X, model):
    labels = model.labels_
    centroids = model.cluster_centers_

    distances = np.linalg.norm(X - centroids[labels], axis=1)

    global_avg = distances.mean()

    per_cluster = []
    for k in range(model.n_clusters):
        mask = labels == k
        per_cluster.append(distances[mask].mean())

    return [global_avg] + per_cluster

avg_distances = average_cluster_distance(X_5_scaled, kmeans_5)
print(f"Global average distance: {avg_distances[0]:.4f}")
for i, d in enumerate(avg_distances[1:]):
    print(f"  Cluster {i}: {d:.4f}")

Global average distance: 0.7702
  Cluster 0: 0.6329
  Cluster 1: 2.0612
  Cluster 2: 5.4806
  Cluster 3: 2.2260
  Cluster 4: 0.8795


<p style='color:red;font-weight:bold'>Exercise 5.3 :</p>

Try using different values for the maximum number of iterations of the *K-Means Algorithm*. Then, report your results and describe them.

In [39]:
max_iter_values = [1, 2, 5, 10, 50, 100, 300, 500]
results_iter = []

for mi in max_iter_values:
    km = KMeans(n_clusters=5, max_iter=mi, random_state=42, n_init=10)
    km.fit(X_5_scaled)
    avg_d = average_cluster_distance(X_5_scaled, km)
    results_iter.append({
        'max_iter': mi,
        'inertia': round(km.inertia_, 2),
        'global_avg_dist': round(avg_d[0], 4),
        'n_iter_actual': km.n_iter_
    })

df_iter = pd.DataFrame(results_iter)
df_iter

,max_iter,inertia,global_avg_dist,n_iter_actual
0,1,13743.57,0.7726,1
1,2,13728.03,0.7701,2
2,5,13727.63,0.7702,4
3,10,13727.63,0.7702,4
4,50,13727.63,0.7702,4
5,100,13727.63,0.7702,4
6,300,13727.63,0.7702,4
7,500,13727.63,0.7702,4


With very few iterations (1-2), K-Means has not yet converged and the inertia / average distance is higher, meaning the clusters are not yet well-formed. As we increase `max_iter`, the algorithm converges quickly — typically within 10-50 iterations for this dataset. Beyond that point, increasing `max_iter` has no effect because the algorithm has already converged (the actual number of iterations `n_iter_actual` stays the same). This shows that K-Means converges relatively fast on this standardized dataset, and setting `max_iter=300` (the default) is more than sufficient.

<p style='color:red;font-weight:bold'>Exercise 5.4 :</p>

Analyze how the category distributions change when clustering with $\text{n\_cluster} = {2,3,4,5}$ and compare the results. Observe how categories are grouped within each cluster and note any significant shifts as the number of clusters increases. Describe the distributions obtained and evaluate the average distances for each $\text{n\_cluster}$.

In [40]:
n_clusters_values = [2, 3, 4, 5]

for n_c in n_clusters_values:
    km = KMeans(n_clusters=n_c, random_state=42, n_init=10)
    km.fit(X_5_scaled)
    df_5[f'Cluster_{n_c}'] = km.labels_

    avg_d = average_cluster_distance(X_5_scaled, km)
    print(f"\n{'='*60}")
    print(f"n_clusters = {n_c} | Global avg distance: {avg_d[0]:.4f}")
    for i, d in enumerate(avg_d[1:]):
        print(f"  Cluster {i}: avg distance = {d:.4f}")

    print(f"\nCategory distribution per cluster (n_clusters={n_c}):")
    ct = pd.crosstab(df_5[f'Cluster_{n_c}'], df_5['Category'])
    print(ct.to_string())

print(f"\n{'='*60}")
print("\nSummary of global average distances:")
for n_c in n_clusters_values:
    km = KMeans(n_clusters=n_c, random_state=42, n_init=10)
    km.fit(X_5_scaled)
    avg_d = average_cluster_distance(X_5_scaled, km)
    print(f"  n_clusters={n_c}: global_avg_dist={avg_d[0]:.4f}, inertia={km.inertia_:.2f}")


n_clusters = 2 | Global avg distance: 1.1003
  Cluster 0: avg distance = 1.0410
  Cluster 1: avg distance = 5.7066

Category distribution per cluster (n_clusters=2):
Category   ART_AND_DESIGN  AUTO_AND_VEHICLES  BEAUTY  BOOKS_AND_REFERENCE  BUSINESS  COMICS  COMMUNICATION  DATING  EDUCATION  ENTERTAINMENT  EVENTS  FAMILY  FINANCE  FOOD_AND_DRINK  GAME  HEALTH_AND_FITNESS  HOUSE_AND_HOME  LIBRARIES_AND_DEMO  LIFESTYLE  MAPS_AND_NAVIGATION  MEDICAL  NEWS_AND_MAGAZINES  PARENTING  PERSONALIZATION  PHOTOGRAPHY  PRODUCTIVITY  SHOPPING  SOCIAL  SPORTS  TOOLS  TRAVEL_AND_LOCAL  VIDEO_PLAYERS  WEATHER
Cluster_2                                                                                                                                                                                                                                                                                                                                                                                                     

**Observations:**

- **n_clusters=2:** The dataset is split into one very large cluster containing the majority of apps and one small cluster with outliers (apps with very high reviews/installs). Most categories are present in both clusters but heavily concentrated in the large one. The global average distance is the highest.

- **n_clusters=3:** A third cluster appears, typically separating apps with moderate popularity from the rest. Categories like GAME and FAMILY, which have a wide range of installs/reviews, start to spread across clusters.

- **n_clusters=4:** Further refinement occurs. We can observe clearer separation between groups — for instance, popular paid apps may form their own cluster. Category distributions become more varied across clusters.

- **n_clusters=5:** The clusters become more specialized. The global average distance decreases further, indicating tighter clusters. However, category distributions remain fairly mixed across clusters because the clustering is based on numerical features (Rating, Reviews, Size, Installs, Price), not on the category itself. Categories like FAMILY and GAME span multiple clusters because they contain apps with very different characteristics.

**Average distances** decrease as n_clusters increases — this is expected since more centroids allow each point to be closer to its assigned center. However, the improvement diminishes with more clusters, suggesting that 4-5 clusters capture the main structure of this dataset well.